## Import Libraries

In [1]:
import pandas as pd
import numpy as np

## Load MIMIC-IV and Hospital Staffing datasets

In [2]:
patients = pd.read_csv("Datasets/patients.csv")
admissions = pd.read_csv("Datasets/admissions.csv")
icustays = pd.read_csv("Datasets/icustays.csv")

staffing = pd.read_csv("Datasets/hospital-staffing-2009-2013.csv")

## Merge Tables

In [3]:
df = icustays.merge(admissions, on=["hadm_id", "subject_id"], how="left")
df = df.merge(patients, on="subject_id", how="left")

In [4]:
print(df.columns)

Index(['subject_id', 'hadm_id', 'stay_id', 'first_careunit', 'last_careunit',
       'intime', 'outtime', 'los', 'admittime', 'dischtime', 'deathtime',
       'admission_type', 'admit_provider_id', 'admission_location',
       'discharge_location', 'insurance', 'language', 'marital_status', 'race',
       'edregtime', 'edouttime', 'hospital_expire_flag', 'gender',
       'anchor_age', 'anchor_year', 'anchor_year_group', 'dod'],
      dtype='object')


## Process Time Features

In [5]:
df['intime'] = pd.to_datetime(df['intime'])
df['outtime'] = pd.to_datetime(df['outtime'])

# ICU length of stay (hours)
df['icu_los_hours'] = (df['outtime'] - df['intime']).dt.total_seconds() / 3600

## Create Operational Features

### Department & Shift

In [6]:
departments = ['ICU', 'Emergency', 'Surgery', 'General Ward']
df['department'] = np.random.choice(departments, size=len(df))

df['shift_type'] = np.where(df['intime'].dt.hour < 18, 'Day', 'Night')

### Workload Index

In [7]:
df['intime_hour'] = df['intime'].dt.floor('H')

hourly_counts = df.groupby('intime_hour')['stay_id'].transform('count')

# Normalization
min_val = hourly_counts.min()
max_val = hourly_counts.max()

if max_val == min_val:
    df['workload_index'] = 50
else:
    df['workload_index'] = 100 * (
        (hourly_counts - min_val) / (max_val - min_val)
    )

print(df['workload_index'].describe())

count    140.0
mean      50.0
std        0.0
min       50.0
25%       50.0
50%       50.0
75%       50.0
max       50.0
Name: workload_index, dtype: float64


/var/folders/35/nzrm0j710w35fmkzxc9_ffc00000gn/T/ipykernel_2339/3737820719.py:1: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  df['intime_hour'] = df['intime'].dt.floor('H')


### Staffing Ratio

In [8]:
# Normalize properly between 0–1
staffing['staffing_ratio'] = (
    staffing['Productive Hours per Adjusted Patient Day'] -
    staffing['Productive Hours per Adjusted Patient Day'].min()
) / (
    staffing['Productive Hours per Adjusted Patient Day'].max() -
    staffing['Productive Hours per Adjusted Patient Day'].min()
)

df['staffing_ratio'] = np.random.choice(
    staffing['staffing_ratio'].dropna(),
    size=len(df)
)

### Bed Occupancy Rate

In [9]:
# Higher productive hours → higher occupancy proxy
staffing['bed_occupancy_rate'] = (
    staffing['Productive Hours'] -
    staffing['Productive Hours'].min()
) / (
    staffing['Productive Hours'].max() -
    staffing['Productive Hours'].min()
)

bed_occupancy = staffing['bed_occupancy_rate'].dropna().values

df['bed_occupancy_rate'] = np.random.choice(bed_occupancy, size=len(df))

### Operational Features

In [10]:
df['overtime_hours'] = np.random.uniform(0, 20, size=len(df))
df['equipment_availability'] = np.random.uniform(0.6, 1.0, size=len(df))
df['incident_last_7_days'] = np.random.poisson(2, size=len(df))

### Patient Acuity

In [11]:
def acuity(los):
    if los > 100:
        return 5
    elif los > 72:
        return 4
    elif los > 48:
        return 3
    elif los > 24:
        return 2
    else:
        return 1

df['patient_acuity'] = df['icu_los_hours'].apply(acuity)

### Response Time

In [12]:
df['avg_response_time'] = np.random.uniform(5, 60, size=len(df))

## Create Risk Target

### Normalize Features

In [13]:
# Normalize everything properly FIRST
df['workload_norm'] = df['workload_index'] / 100
df['overtime_norm'] = df['overtime_hours'] / 20
df['acuity_norm'] = df['patient_acuity'] / 5
df['response_norm'] = df['avg_response_time'] / 60

# Fix staffing + equipment risks
df['staffing_risk'] = (1 - df['staffing_ratio']).clip(0, 1)
df['equipment_risk'] = (1 - df['equipment_availability']).clip(0, 1)

# Fix incidents (scale properly)
df['incident_norm'] = df['incident_last_7_days'] / 5

# FIX occupancy (force realistic scale)
df['occupancy_norm'] = np.clip(df['bed_occupancy_rate'] * 100, 0, 1)

### Risk Score

In [14]:
df['risk_score'] = (
    0.20 * df['staffing_risk'] +
    0.15 * df['workload_norm'] +
    0.15 * df['overtime_norm'] +
    0.10 * df['equipment_risk'] +
    0.10 * df['incident_norm'] +
    0.10 * df['acuity_norm'] +
    0.10 * df['response_norm'] +
    0.10 * df['occupancy_norm']
)

df['risk_score'] += np.random.normal(0, 0.05, len(df))

### Add Shift + Department Effects

In [15]:
# Shift effect
df['risk_score'] += df['shift_type'].apply(lambda x: 0.05 if x == 'Night' else 0)

# Department effect
dept_risk = {
    'ICU': 0.10,
    'Emergency': 0.08,
    'Surgery': 0.05,
    'General Ward': 0.03
}

df['risk_score'] += df['department'].map(dept_risk)

### Final Target

In [16]:
# Add small randomness to avoid ties / flat distribution
df['risk_score'] += np.random.normal(0, 0.05, len(df))

# Dynamic threshold (top 30% = high risk)
threshold = df['risk_score'].quantile(0.7)

df['target'] = (df['risk_score'] > threshold).astype(int)

print("Threshold:", threshold)
print(df[['risk_score']].describe())
print(df['target'].value_counts(normalize=True))

Threshold: 0.667844282976277
       risk_score
count  140.000000
mean     0.608754
std      0.094502
min      0.346670
25%      0.542982
50%      0.613638
75%      0.674982
max      0.824585
target
0    0.7
1    0.3
Name: proportion, dtype: float64


### Final Data Frame

In [17]:
final_df = df[
    [
        'department',
        'staffing_ratio',
        'workload_index',
        'overtime_hours',
        'equipment_availability',
        'incident_last_7_days',
        'shift_type',
        'patient_acuity',
        'avg_response_time',
        'bed_occupancy_rate',
        'target'
    ]
]

## Final Dataset

In [18]:
final_df.to_csv("operational_risk_dataset.csv", index=False)

print(final_df.head())
print(final_df['target'].value_counts())

     department  staffing_ratio  workload_index  overtime_hours  \
0  General Ward        0.118464              50       18.792862   
1           ICU        0.000000              50        2.970016   
2           ICU        0.022564              50        2.540768   
3           ICU        0.006402              50       10.579799   
4     Emergency        0.030741              50       19.585332   

   equipment_availability  incident_last_7_days shift_type  patient_acuity  \
0                0.647637                     2      Night               5   
1                0.843126                     2        Day               5   
2                0.938980                     2      Night               1   
3                0.871441                     2        Day               4   
4                0.873443                     3      Night               2   

   avg_response_time  bed_occupancy_rate  target  
0          16.806574            0.000259       1  
1           7.814826      